# 02 - Set Model Constrains

This notebook sets the constraints of the GEM to mimic medium the nutrients provided by the modified Rhodospirillaceae medium used by Montiel-Corona and Buitron.


## Installation

In [1]:
# Install cobrapy
!pip install -qq cobra catboost

In [2]:
# Check installation
!pip show cobra

Name: cobra
Version: 0.30.0
Summary: COBRApy is a package for constraint-based modeling of metabolic networks.
Home-page: https://opencobra.github.io/cobrapy
Author: The cobrapy core development team.
Author-email: cobra-pie@googlegroups.com
License: LGPL-2.0-or-later OR GPL-2.0-or-later
Location: /usr/local/lib/python3.12/dist-packages
Requires: appdirs, depinfo, diskcache, future, httpx, numpy, optlang, pandas, pydantic, python-libsbml, rich, ruamel.yaml, swiglpk
Required-by: 


## Mount drive

In [3]:
import os
from pathlib import Path
from google.colab import drive

def mount_drive():
  drive.mount('/content/drive', force_remount=True)
  drive_folder = "metabolic_modelling/phb-optimization-rpalustris/"
  os.chdir('/content/drive/MyDrive/'+ drive_folder)
  global PROJECT_ROOT
  PROJECT_ROOT = Path(os.getcwd())

mount_drive()


Mounted at /content/drive


## Load packages

In [4]:
# load packages

import logging
import cobra
from cobra import Reaction, Metabolite, Model, io
from cobra.io import (
    load_model, load_json_model, save_json_model,
    load_matlab_model, save_matlab_model,
    read_sbml_model, write_sbml_model
)

from cobra.flux_analysis import flux_variability_analysis, pfba

from cobra.util.solver import linear_reaction_coefficients

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools
from itertools import product
import seaborn as sns
import json
import math
import copy
from tqdm import tqdm


## Set paths

In [5]:
# Import path saved in src

import sys
sys.path.append(str(PROJECT_ROOT / "src"))
from src.paths import MODELS_DIR, PHB_MODEL_DIR, PHB_CHECKPOINTS_DIR, PHB_RESULTS_DIR


## Load .xml model

In [7]:
# Load model

# Full path to the SBML file saved in notebook 01_model_setup_phb.pynb
phb_model_path = PHB_MODEL_DIR / "01_model_rpalustris_PHB.xml"

# Load model
model_phb = io.read_sbml_model(
    str(phb_model_path),
    use_fbc=False
)

print("Model loaded successfully")

Model loaded successfully


## Check model with PHBS_syn as objective

In [8]:
# Load model
model_phb

Name,iDT1294
Memory address,7c95164908f0
Number of metabolites,2124
Number of reactions,2721
Number of genes,1294
Number of groups,118
Objective expression,1.0*PHBS_syn - 1.0*PHBS_syn_reverse_8e587
Compartments,"c, u, p, e"


In [9]:
# Check model summary
model_phb.summary() # PHBS_syn should be the objective function with UB = 0.3978

Metabolite,Reaction,Flux,C-Number,C-Flux
ac_e,EX_ac_e,0.0001,2,0.01%
actn__R_e,EX_actn__R_e,0.3979,4,99.99%
o2_e,EX_o2_e,5.556E-06,0,0.00%
phbg_c,SK_phbg_c,0.3978,0,0.00%
Metabolite,Reaction,Flux,C-Number,C-Flux
PHB_c,DM_PHB_c,-0.3978,4,99.97%
h2_c,DM_h2_c,-0.3979,0,0.00%
h2o_e,EX_h2o_e,-1.111E-05,0,0.00%
2obut_c,sink_2obut_c,-0.0001,4,0.03%


## Copy model

In [28]:
# Copy phb model
model_phb2 = model_phb.copy()

# Medium Montiel-Corona 2022

Here, experimental medium from Montiel-Corona and Buitron 2022 is mimicked as constraints for "EX" reactions.

Note:
1.   Photon reactions are OFF by default.
2.   Wavelengths >700 nm missing.

In [31]:
# Set medium as specified in Montiel-Corona & Buitrón 2022.

# Fluxes from medium composition were calculated using the highest PHB production reported in Montiel-Corona et al., 2022
# Forumal used q_x = total amount in medium/ biomass x time
# These parameters are Biomas: 1367, Time: 3 days, PHB % W/W, C/N 12 (Montiel-Corona et al., 2022)
# === CONSERVATIVE UPPER-BOUND UPTAKE LIMITS (from medium composition) ===
# These are maximal possible average uptake rates (mmol·gDW⁻¹·h⁻¹)
# computed from the medium composition, assuming full consumption over 72 h.
# They are safe, conservative constraints.

nh4_from_acetate = 0.06590  # ammonium from NH4-acetate
nh4_from_nh4cl = 0.07598    # ammonium from NH4Cl
total_nh4_uptake = nh4_from_acetate + nh4_from_nh4cl

medium_montiel_buitron_2022_conservative_upperbound = {
    "EX_ac_e":     0.2477,        # acetate (but we override with q_ac =0.1209 later)
    # Cofeeding
    #"EX_fum_e":    1,       # test: fumarate increases production a lot
    #"EX_hxa_e":    1,       # test: hexanoate
    #"EX_ppa_e":    1,       # test: propionate does not help much
    # Other nutrients
    "EX_nh4_e":   total_nh4_uptake,    # ammonium (from NH4-acetate & NH4Cl)
    "EX_pi_e":     0.03733,       # phosphate (KH2PO4)
    "EX_mg2_e":    0.01604,       # magnesium (MgSO4)
    "EX_na1_e":    0.06955,       # sodium (NaCl)
    "EX_ca2_e":    0.003455,      # calcium
    "EX_fe3_e":    0.0004147,     # iron(III)
    #"EX_cys__L_e": 0.01933,       # L-cysteine·HCl
    #"EX_thm_e":    0.00003363,    # thiamine·HCl
    #"EX_nac_e":    0.00008250,    # nicotinic acid
    "EX_zn2_e":    0.000002337,   # Zn
    "EX_mn2_e":    0.000000620,   # Mn
    "EX_bo3_e":    0.00001972,    # Boron
    #"EX_co2pp_e":  0.000003414,   # Cobalt (ID guessed)
    "EX_cu2_e":    0.000000234,   # Copper
    "EX_ni2_e":    0.000000345,   # Nickel
    "EX_mobd_e":   0.000000559,   # molybdate (NaMoO4)
    "EX_co2_e":    0.0,          # CO2
    "EX_o2_e":     0.0,          # set to 0 if anaerobic; set to >0 if aerobic
    "EX_hco3_e":   0.0,          # bicarbonate

    # Connect Photons (all wavelenghts to simulate outdoors raceway bioreactor)
    'EX_photon410_e': 1000,
    'EX_photon430_e': 1000,
    'EX_photon450_e': 1000,
    'EX_photon470_e': 1000,
    'EX_photon490_e': 1000,
    'EX_photon510_e': 1000,
    'EX_photon530_e': 1000,
    'EX_photon550_e': 1000,
    'EX_photon570_e': 1000,
    'EX_photon590_e': 1000,
    'EX_photon610_e': 1000,
    'EX_photon630_e': 1000,
    'EX_photon650_e': 1000,
    'EX_photon670_e': 1000,
    'EX_photon690_e': 1000
}

# Close ALL uptake (sets a clean minimal medium)
for ex in model_phb2.exchanges:
    ex.lower_bound = 0.0

# Apply medium_montiel_buitron_2022_conservative_upperbound
print("\n=== Applying conservative medium upper-bound uptake limits ===")
for rxn_id, max_uptake in medium_montiel_buitron_2022_conservative_upperbound.items():
    if rxn_id in model_phb2.reactions:
        model_phb2.reactions.get_by_id(rxn_id).lower_bound = -max_uptake
        print(f"{rxn_id}: lower_bound = {-max_uptake:.6g}")
    else:
        print(f"{rxn_id}: NOT FOUND IN MODEL (skipped)")



=== Applying conservative medium upper-bound uptake limits ===
EX_ac_e: lower_bound = -0.2477
EX_nh4_e: lower_bound = -0.14188
EX_pi_e: lower_bound = -0.03733
EX_mg2_e: lower_bound = -0.01604
EX_na1_e: lower_bound = -0.06955
EX_ca2_e: lower_bound = -0.003455
EX_fe3_e: lower_bound = -0.0004147
EX_zn2_e: lower_bound = -2.337e-06
EX_mn2_e: lower_bound = -6.2e-07
EX_bo3_e: lower_bound = -1.972e-05
EX_cu2_e: lower_bound = -2.34e-07
EX_ni2_e: lower_bound = -3.45e-07
EX_mobd_e: lower_bound = -5.59e-07
EX_co2_e: lower_bound = -0
EX_o2_e: lower_bound = -0
EX_hco3_e: lower_bound = -0
EX_photon410_e: lower_bound = -1000
EX_photon430_e: lower_bound = -1000
EX_photon450_e: lower_bound = -1000
EX_photon470_e: lower_bound = -1000
EX_photon490_e: lower_bound = -1000
EX_photon510_e: lower_bound = -1000
EX_photon530_e: lower_bound = -1000
EX_photon550_e: lower_bound = -1000
EX_photon570_e: lower_bound = -1000
EX_photon590_e: lower_bound = -1000
EX_photon610_e: lower_bound = -1000
EX_photon630_e: lower_

# Save medium

In [35]:
# Save medium medium_montiel_buitron_2022_conservative_upperbound

# Collect bounds for all exchange reactions
rows = []
for rxn in model_phb2.exchanges:
    rows.append({
        "Reaction": rxn.id,
        "Name": rxn.name,
        "LowerBound": rxn.lower_bound,
        "UpperBound": rxn.upper_bound
    })

# Convert to DataFrame
medium_df = pd.DataFrame(rows)

# Save to Excel
medium_df.to_csv(PHB_CHECKPOINTS_DIR / "02_model_phb_medium_montiel_buitron_2022.csv", index=False)

print('Medium Montiel-Buitron saved successfully')


Medium Montiel-Buitron saved successfully


## Check PHB reaction


In [33]:
# Check reaction directly associated to PHB production
phb_rxn = model_phb.reactions.get_by_id("PHBS_syn")
print(phb_rxn.bounds)
print(phb_rxn.reaction)

(0.0, 0.3978)
3hbcoa__R_c + phbg_c --> PHB_c + coa_c


## Check biomass reaction bounds

In [34]:
# Check reaction EX_ac_e
biomass_rxn = model_phb.reactions.get_by_id('BIOMASS__1')
print(biomass_rxn.bounds)
print(biomass_rxn.reaction)

(0.0, 0.0)
30.0 atp_c + 0.0977 bm_carbs_c + 0.00119 bm_cofactors_c + 0.0795 bm_cw_c + 0.0073 bm_dna_c + 0.01 bm_min_c + 0.159 bm_oth_c + 0.0197 bm_pigm_c + 0.5112 bm_pro_c + 0.1136 bm_rna_c + 30.0 h2o_c --> 30.0 adp_c + 30.0 h_c + 30.0 pi_c


# Save all reactions

To snapshot reactions before performing optimization


In [36]:
# Collect all reactions in a list of dicts
rxn_data = []
for rxn in model_phb2.reactions:
    rxn_data.append({
        "ReactionID": rxn.id,
        "Name": rxn.name,
        "Equation": rxn.build_reaction_string(use_metabolite_names=True),
        "LowerBound": rxn.lower_bound,
        "UpperBound": rxn.upper_bound,
        "ObjectiveCoeff": rxn.objective_coefficient,
        "Subsystem": rxn.subsystem
    })

# Convert to DataFrame
rxn_df = pd.DataFrame(rxn_data)

# Save to Excel
rxn_df.to_csv(PHB_CHECKPOINTS_DIR / "02_model_phb_all_reactions_bounds.csv", index=False)

print("Saved complete reaction table to model_all_reaction_bounds.csv")


Saved complete reaction table to model_all_reaction_bounds.csv


# Save model phb ready for FBA

In [38]:
# Save model with phbvv reactions in phb results directory

# Target file
phb_model_file = PHB_MODEL_DIR / "02_model_rpalustris_PHB_constrained.xml"

# Save COBRA model as .xml file
cobra.io.write_sbml_model(model_phb2, str(phb_model_file))